# 04 — RAG Pipeline (end-to-end smoke test)

Loads each of the 4 configs (A/B/C/D) and runs a handful of representative Vietnamese tax-law questions through them. Use this notebook on Kaggle once the LoRA adapter has been pushed to `Tamir39/qwen2_5-7b-vietnam-tax-lora`.

**Setup**: same as `03_finetune_lora_kaggle.ipynb` — GPU on, internet on, HF_TOKEN secret optional (only needed if the adapter repo is private).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

# Build the FAISS index once if it doesn't exist yet (Phase 2 artifact).
from src.config import INDEX_DIR
if not (INDEX_DIR / "kb.faiss").exists():
    import subprocess
    subprocess.check_call([sys.executable, str(ROOT / "scripts" / "build_index.py")])
print("index ready:", (INDEX_DIR / "kb.faiss").exists())

In [ ]:
from src.inference.pipeline import InferencePipeline

CFGS = ROOT / "experiments" / "configs"
configs = {
    "A": CFGS / "A_base_no_rag.yaml",
    "B": CFGS / "B_base_with_rag.yaml",
    "C": CFGS / "C_finetuned_no_rag.yaml",
    "D": CFGS / "D_finetuned_with_rag.yaml",
}

questions = [
    "Thuế suất thuế giá trị gia tăng đối với hàng hóa xuất khẩu là bao nhiêu?",
    "Đối tượng nào được miễn thuế sử dụng đất phi nông nghiệp?",
    "Hàng hóa nào chịu thuế tiêu thụ đặc biệt?",
]

# Load configs sequentially because each one needs ~14 GB GPU RAM.
for key, path in configs.items():
    print(f"\n========== Config {key}: {path.name} ==========")
    pipe = InferencePipeline(path)
    pipe.load()
    for q in questions:
        res = pipe.answer(q)
        print(f"\nQ: {q}")
        print(f"A: {res.answer}")
        if res.retrieved_passage_ids:
            print(f"  retrieved: {res.retrieved_passage_ids}")
    del pipe
    import gc, torch
    gc.collect()
    torch.cuda.empty_cache()